In [1]:
!pip install -q fastapi uvicorn pyngrok nest-asyncio

In [2]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries for our entire adventure ---
import os
import re
import asyncio
from IPython.display import display, Markdown
import google.generativeai as genai
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search, ToolContext
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

print("✅ All libraries are ready to go!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


✅ All libraries are ready to go!


In [11]:
# --- Securely Configure Your API Key ---

# Prompt the user for their API key securely
api_key = getpass('get api')

# Get Your API Key HERE 👉 https://codelabs.developers.google.com/onramp/instructions#0
# Configure the generative AI library with the provided key
genai.configure(api_key=api_key)

# Set the API key as an environment variable for ADK to use
os.environ['GOOGLE_API_KEY'] = api_key

print("✅ API Key configured successfully! Let the fun begin.")

get api··········
✅ API Key configured successfully! Let the fun begin.


In [4]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [5]:
!pip install google-adk --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.5/584.5 kB 21.9 MB/s eta 0:00:00
  Attempting uninstall: google-adk
    Found existing installation: google-adk 1.26.0
    Uninstalling google-adk-1.26.0:
      Successfully uninstalled google-adk-1.26.0


In [6]:
# --- Agent Definitions for MycoGuard Specialist Team ---


# 1. data_auditor_agent
data_auditor_agent = Agent(
    name="data_auditor_agent",
    model="gemini-2.5-flash",
    description="Agent specialized in cleaning data, handling missing values, and encoding categorical variables.",
    instruction="""
    You are the MycoGuard Data Auditor 🧹 - a specialized AI assistant that prepares raw CSV data for machine learning.

    Your Mission:
    Write the exact Python code to load and clean the 'mushroom(1).csv' dataset.

    Guidelines:
    1. Write code to import the 'pandas' and 'sklearn.preprocessing' libraries.
    2. Write code to read 'mushroom(1).csv' into a pandas DataFrame.
    3. Write code to check for missing values.
    4. Write code to use LabelEncoder to convert all categorical variables into numeric values.
    5. Output the complete, well-commented Python code snippet.
    """
)

# 2. classifier_forge_agent
classifier_forge_agent = Agent(
    name="classifier_forge_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are the MycoGuard Classifier Forge 🧠 - an expert quantitative data scientist.

    Your Mission:
    Based on the cleaning steps from the Auditor, write the Python code to train a Random Forest classifier to predict the 'class' column.

    Guidelines:
    1. Write code to split the data into training and testing sets.
    2. Write code to train a RandomForestClassifier.
    3. Write code to find the top 5 most important features and plot them as a horizontal bar chart.
    4. Output the complete, well-commented Python code snippet.
    """
)

# 3. safety_validator_agent
safety_validator_agent = Agent(
    name="safety_validator_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are the MycoGuard Safety Validator 🛡️ - a strict AI Safety auditor.

    Your Mission:
    Based on the model training code from the Forge, write the Python code to perform a strict safety audit.

    Guidelines:
    1. Write code to generate a Confusion Matrix.
    2. Write code to specifically calculate the False Negative Rate (poisonous mushrooms misclassified as edible).
    3. Add print statements in the code to issue a severe warning if the False Negative Rate is greater than 0.
    4. Output the complete, well-commented Python code snippet.
    """
)

# --- The Brain of the Operation: The Router Agent ---
router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""
    You are a request router. Your job is to analyze a user's query and decide which of the following agents or workflows is best suited to handle it.
    Do not answer the query yourself, only return the name of the most appropriate choice.

    Available Options:
    - 'data_auditor_agent': For queries about cleaning data or writing preprocessing code.
    - 'classifier_forge_agent': For queries about writing model training or plotting code.
    - 'safety_validator_agent': For queries about writing AI safety audit code.
    - 'mycoguard_sequential_combo': Use this for complex queries that ask to sequentially write the code for data cleaning, model training, and safety validation.

    Only return the single, most appropriate option's name and nothing else.
    """
)

worker_agents = {
    "data_auditor_agent": data_auditor_agent,
    "classifier_forge_agent": classifier_forge_agent,
    "safety_validator_agent": safety_validator_agent,
}

print("🍄 MycoGuard Agent team assembled successfully! Ready to generate quantitative code.")

🍄 MycoGuard Agent team assembled successfully! Ready to generate quantitative code.


In [12]:
import asyncio
from IPython.display import display, Markdown

# --- A Helper Function to Run Our Agents ---
async def run_agent_query(agent, query: str, session, user_id: str, is_router: bool = False):
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:

                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print("\n" + "-"*50)
        print("✅ Final Generated Code/Response:")
        display(Markdown(final_response))
        print("-"*50 + "\n")
    return final_response

# --- Initialize our Session Service ---
session_service = InMemorySessionService()
my_user_id = "mycoguard_admin_001"

# --- Let's Test the Sequential Workflow! ---
async def run_sequential_app():

    query = "Design the complete Python architecture to analyze the mushroom dataset: clean the data, train a random forest model, and validate the safety."

    print(f"\n{'='*60}\n🗣️ Processing New Query: '{query}'\n{'='*60}")

    # 1. Router Agent
    router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
    print("🧠 Asking the router agent to make a decision...")
    chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)
    chosen_route = chosen_route.strip().replace("'", "")
    print(f"🚦 Router has selected route: '{chosen_route}'")

    if chosen_route == 'mycoguard_sequential_combo':
        print("\n--- 🚀 Starting MycoGuard Generative Combo Workflow ---")


        print("\n[Step 1] Auditor Agent is drafting data preprocessing code...")
        auditor_query = "Please write the Python code to load 'mushroom(1).csv', handle missing values, and encode categorical variables using LabelEncoder."
        auditor_session = await session_service.create_session(app_name=data_auditor_agent.name, user_id=my_user_id)
        auditor_code = await run_agent_query(data_auditor_agent, auditor_query, auditor_session, my_user_id)

        print("\n[Step 2] Forge Agent is drafting Random Forest training code...")
        forge_query = f"Based on this preprocessing code:\n{auditor_code}\nWrite the next part of the script to train a Random Forest classifier and plot the top 5 feature importances."
        forge_session = await session_service.create_session(app_name=classifier_forge_agent.name, user_id=my_user_id)
        forge_code = await run_agent_query(classifier_forge_agent, forge_query, forge_session, my_user_id)


        print("\n[Step 3] Safety Validator Agent is drafting the strict audit code...")
        validator_query = f"Based on the training code:\n{forge_code}\nWrite the final part of the script to output a Confusion Matrix and strictly check for False Negatives (poisonous misclassified as edible)."
        validator_session = await session_service.create_session(app_name=safety_validator_agent.name, user_id=my_user_id)
        await run_agent_query(safety_validator_agent, validator_query, validator_session, my_user_id)

        print("\n--- 🍄 MycoGuard Generative Workflow Complete ---")
    else:
        print(f"🚨 Unexpected route chosen: {chosen_route}")


await run_sequential_app()


🗣️ Processing New Query: 'Design the complete Python architecture to analyze the mushroom dataset: clean the data, train a random forest model, and validate the safety.'
🧠 Asking the router agent to make a decision...

🚀 Running query for agent: 'router_agent' in session: '2dfb6111-e427-40d6-9583-531822a79dcb'...
🚦 Router has selected route: 'mycoguard_sequential_combo'

--- 🚀 Starting MycoGuard Generative Combo Workflow ---

[Step 1] Auditor Agent is drafting data preprocessing code...

🚀 Running query for agent: 'data_auditor_agent' in session: 'ef9adaff-6b85-4796-a563-b22e12351b29'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll exp

```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll explicitly replace them with NaN during loading or shortly after.
print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print(f"Initial shape: {df.shape}")
print("\nFirst 5 rows of the dataset:")
print(df.head())

# --- 2. Handle missing values ---
# The mushroom dataset is known for using '?' as a missing value indicator,
# particularly in the 'stalk-root' column.
# First, replace '?' with numpy's NaN for proper missing value detection.
print("\nChecking and handling '?' values...")
df.replace('?', np.nan, inplace=True)

# Now, check for any standard missing values (NaN)
missing_values_before_handling = df.isnull().sum()
print("\nMissing values per column (after replacing '?'):")
print(missing_values_before_handling[missing_values_before_handling > 0])

# For the mushroom dataset, the 'stalk-root' column typically has missing values.
# A common strategy is to drop rows with missing 'stalk-root' values,
# as it's a relatively small portion and helps in maintaining data integrity
# without introducing bias through imputation in this specific context.
initial_rows = df.shape[0]
df.dropna(inplace=True)
rows_after_dropna = df.shape[0]
print(f"\nDropped {initial_rows - rows_after_dropna} rows with missing values.")
print(f"Shape after dropping missing values: {df.shape}")

# Verify no more missing values
print("\nMissing values per column (after dropping rows):")
print(df.isnull().sum().sum()) # Should be 0 if all handled

# --- 3. Encode categorical variables using LabelEncoder ---
# All columns in the mushroom dataset are categorical and need to be converted
# into numerical format for machine learning algorithms.
print("\nEncoding categorical variables using LabelEncoder...")
label_encoder = LabelEncoder()

# Iterate over each column and apply LabelEncoder
for column in df.columns:
    # Check if the column is of object type (string/categorical)
    # Even if it's not object type, LabelEncoder can still process it,
    # but it's good practice to ensure it's categorical.
    if df[column].dtype == 'object':
        df[column] = label_encoder.fit_transform(df[column])
        print(f"Encoded column: '{column}'")
    else:
        # This case is unlikely for the standard mushroom dataset, but good for robustness
        print(f"Column '{column}' is already numeric or not 'object' type, skipping encoding.")

print("\nAll categorical variables encoded successfully.")

# --- 4. Display the processed dataset ---
print("\nFirst 5 rows of the processed dataset:")
print(df.head())
print(f"\nProcessed dataset shape: {df.shape}")
print("\nData types of the processed dataset:")
print(df.dtypes)
```

--------------------------------------------------


[Step 2] Forge Agent is drafting Random Forest training code...

🚀 Running query for agent: 'classifier_forge_agent' in session: 'efca458d-bf47-414d-871a-905392b43646'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll explicitly replace them with NaN during loading or shortly after.
print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print(f"Initial shape: {df.shape}")
prin

```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll explicitly replace them with NaN during loading or shortly after.
print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print(f"Initial shape: {df.shape}")
print("\nFirst 5 rows of the dataset:")
print(df.head())

# --- 2. Handle missing values ---
# The mushroom dataset is known for using '?' as a missing value indicator,
# particularly in the 'stalk-root' column.
# First, replace '?' with numpy's NaN for proper missing value detection.
print("\nChecking and handling '?' values...")
df.replace('?', np.nan, inplace=True)

# Now, check for any standard missing values (NaN)
missing_values_before_handling = df.isnull().sum()
print("\nMissing values per column (after replacing '?'):")
print(missing_values_before_handling[missing_values_before_handling > 0])

# For the mushroom dataset, the 'stalk-root' column typically has missing values.
# A common strategy is to drop rows with missing 'stalk-root' values,
# as it's a relatively small portion and helps in maintaining data integrity
# without introducing bias through imputation in this specific context.
initial_rows = df.shape[0]
df.dropna(inplace=True)
rows_after_dropna = df.shape[0]
print(f"\nDropped {initial_rows - rows_after_dropna} rows with missing values.")
print(f"Shape after dropping missing values: {df.shape}")

# Verify no more missing values
print("\nMissing values per column (after dropping rows):")
print(df.isnull().sum().sum()) # Should be 0 if all handled

# --- 3. Encode categorical variables using LabelEncoder ---
# All columns in the mushroom dataset are categorical and need to be converted
# into numerical format for machine learning algorithms.
print("\nEncoding categorical variables using LabelEncoder...")
label_encoder = LabelEncoder()

# Iterate over each column and apply LabelEncoder
for column in df.columns:
    # Check if the column is of object type (string/categorical)
    # Even if it's not object type, LabelEncoder can still process it,
    # but it's good practice to ensure it's categorical.
    if df[column].dtype == 'object':
        df[column] = label_encoder.fit_transform(df[column])
        print(f"Encoded column: '{column}'")
    else:
        # This case is unlikely for the standard mushroom dataset, but good for robustness
        print(f"Column '{column}' is already numeric or not 'object' type, skipping encoding.")

print("\nAll categorical variables encoded successfully.")

# --- 4. Display the processed dataset ---
print("\nFirst 5 rows of the processed dataset:")
print(df.head())
print(f"\nProcessed dataset shape: {df.shape}")
print("\nData types of the processed dataset:")
print(df.dtypes)

# --- 5. Prepare data for classification ---
print("\nPreparing data for classification...")
# Define features (X) and target (y)
# The 'class' column is our target variable
X = df.drop('class', axis=1)
y = df['class']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFirst 5 rows of features (X):")
print(X.head())
print("\nFirst 5 rows of target (y):")
print(y.head())

# --- 6. Split the data into training and testing sets ---
# Using a 80/20 split for training and testing
# stratify=y ensures that both training and testing sets have a proportional
# representation of the target variable's classes.
# random_state for reproducibility.
print("\nSplitting data into training and testing sets (80/20 split)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# --- 7. Train a RandomForestClassifier ---
print("\nTraining a RandomForestClassifier...")
# Initialize the RandomForestClassifier
# n_estimators: number of trees in the forest
# random_state for reproducibility
# class_weight='balanced' can be useful for imbalanced datasets (though not strictly needed here)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Train the model on the training data
model.fit(X_train, y_train)
print("RandomForestClassifier trained successfully.")

# --- 8. Find and plot the top 5 most important features ---
print("\nFinding top 5 most important features...")
# Get feature importances from the trained model
feature_importances = model.feature_importances_

# Create a pandas Series for better visualization and sorting
features_df = pd.Series(feature_importances, index=X.columns)

# Sort features by importance in descending order and get the top 5
top_5_features = features_df.nlargest(5)

print("\nTop 5 most important features:")
print(top_5_features)

# Plot the top 5 most important features as a horizontal bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_features.values, y=top_5_features.index, palette='viridis')
plt.title('Top 5 Most Important Features for Mushroom Classification')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()
print("Top 5 feature importances plotted successfully.")
```

--------------------------------------------------


[Step 3] Safety Validator Agent is drafting the strict audit code...

🚀 Running query for agent: 'safety_validator_agent' in session: '38dbd70c-d498-4b3a-a3dc-0ea17fb5f8ae'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll explicitly replace them with NaN during loading or shortly after.
print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)
prin

```python
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Load the dataset ---
# Define the path to the CSV file
file_path = 'mushroom(1).csv'

# Load the dataset into a pandas DataFrame
# The mushroom dataset often uses '?' to denote missing values,
# so we'll explicitly replace them with NaN during loading or shortly after.
print(f"Loading dataset from: {file_path}")
df = pd.read_csv(file_path)
print("Dataset loaded successfully.")
print(f"Initial shape: {df.shape}")
print("\nFirst 5 rows of the dataset:")
print(df.head())

# --- 2. Handle missing values ---
# The mushroom dataset is known for using '?' as a missing value indicator,
# particularly in the 'stalk-root' column.
# First, replace '?' with numpy's NaN for proper missing value detection.
print("\nChecking and handling '?' values...")
df.replace('?', np.nan, inplace=True)

# Now, check for any standard missing values (NaN)
missing_values_before_handling = df.isnull().sum()
print("\nMissing values per column (after replacing '?'):")
print(missing_values_before_handling[missing_values_before_handling > 0])

# For the mushroom dataset, the 'stalk-root' column typically has missing values.
# A common strategy is to drop rows with missing 'stalk-root' values,
# as it's a relatively small portion and helps in maintaining data integrity
# without introducing bias through imputation in this specific context.
initial_rows = df.shape[0]
df.dropna(inplace=True)
rows_after_dropna = df.shape[0]
print(f"\nDropped {initial_rows - rows_after_dropna} rows with missing values.")
print(f"Shape after dropping missing values: {df.shape}")

# Verify no more missing values
print("\nMissing values per column (after dropping rows):")
print(df.isnull().sum().sum()) # Should be 0 if all handled

# --- 3. Encode categorical variables using LabelEncoder ---
# All columns in the mushroom dataset are categorical and need to be converted
# into numerical format for machine learning algorithms.
print("\nEncoding categorical variables using LabelEncoder...")
label_encoder = LabelEncoder()

# Iterate over each column and apply LabelEncoder
for column in df.columns:
    # Check if the column is of object type (string/categorical)
    # Even if it's not object type, LabelEncoder can still process it,
    # but it's good practice to ensure it's categorical.
    if df[column].dtype == 'object':
        df[column] = label_encoder.fit_transform(df[column])
        print(f"Encoded column: '{column}'")
    else:
        # This case is unlikely for the standard mushroom dataset, but good for robustness
        print(f"Column '{column}' is already numeric or not 'object' type, skipping encoding.")

print("\nAll categorical variables encoded successfully.")

# --- 4. Display the processed dataset ---
print("\nFirst 5 rows of the processed dataset:")
print(df.head())
print(f"\nProcessed dataset shape: {df.shape}")
print("\nData types of the processed dataset:")
print(df.dtypes)

# --- 5. Prepare data for classification ---
print("\nPreparing data for classification...")
# Define features (X) and target (y)
# The 'class' column is our target variable
X = df.drop('class', axis=1)
y = df['class']

print(f"Features (X) shape: {X.shape}")
print(f"Target (y) shape: {y.shape}")
print("\nFirst 5 rows of features (X):")
print(X.head())
print("\nFirst 5 rows of target (y):")
print(y.head())

# --- 6. Split the data into training and testing sets ---
# Using a 80/20 split for training and testing
# stratify=y ensures that both training and testing sets have a proportional
# representation of the target variable's classes.
# random_state for reproducibility.
print("\nSplitting data into training and testing sets (80/20 split)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# --- 7. Train a RandomForestClassifier ---
print("\nTraining a RandomForestClassifier...")
# Initialize the RandomForestClassifier
# n_estimators: number of trees in the forest
# random_state for reproducibility
# class_weight='balanced' can be useful for imbalanced datasets (though not strictly needed here)
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')

# Train the model on the training data
model.fit(X_train, y_train)
print("RandomForestClassifier trained successfully.")

# --- 8. Find and plot the top 5 most important features ---
print("\nFinding top 5 most important features...")
# Get feature importances from the trained model
feature_importances = model.feature_importances_

# Create a pandas Series for better visualization and sorting
features_df = pd.Series(feature_importances, index=X.columns)

# Sort features by importance in descending order and get the top 5
top_5_features = features_df.nlargest(5)

print("\nTop 5 most important features:")
print(top_5_features)

# Plot the top 5 most important features as a horizontal bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=top_5_features.values, y=top_5_features.index, palette='viridis')
plt.title('Top 5 Most Important Features for Mushroom Classification')
plt.xlabel('Feature Importance')
plt.ylabel('Feature Name')
plt.tight_layout() # Adjust layout to prevent labels from overlapping
plt.show()
print("Top 5 feature importances plotted successfully.")

# --- MYCOGUARD SAFETY AUDIT ---
# This section performs a strict safety audit for mushroom classification.
# It focuses on ensuring that poisonous mushrooms are NOT misclassified as edible.

print("\n" + "="*80)
print("--- Starting MycoGuard Safety Audit ---".center(80))
print("="*80)

# 9. Make predictions on the test set
print("\nMaking predictions on the test set for safety audit...")
y_pred = model.predict(X_test)

# 10. Generate and display the Confusion Matrix
print("\nGenerating Confusion Matrix...")
# The 'class' column was encoded using LabelEncoder.
# Typically, 'e' (edible) comes before 'p' (poisonous) alphabetically,
# so 'e' is encoded as 0 and 'p' is encoded as 1.
# Confusion Matrix structure (rows=True, columns=Predicted):
# [[True Negative (TN), False Positive (FP)],   (Actual Edible)
#  [False Negative (FN), True Positive (TP)]]   (Actual Poisonous)
# Where:
# TN: Actual Edible (0), Predicted Edible (0)
# FP: Actual Edible (0), Predicted Poisonous (1)
# FN: Actual Poisonous (1), Predicted Edible (0)  <-- CRITICAL FOR SAFETY
# TP: Actual Poisonous (1), Predicted Poisonous (1)

cm = confusion_matrix(y_test, y_pred)

# Define labels for the confusion matrix for clarity
class_labels = ['Edible (0)', 'Poisonous (1)']

print("\nConfusion Matrix:")
print(cm)

# Plotting the Confusion Matrix for better visualization
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('MycoGuard Safety Audit: Confusion Matrix')
plt.show()

# 11. Specifically calculate the False Negative Rate (FNR)
print("\nCalculating False Negative Rate (FNR) for poisonous mushrooms...")

# False Negatives (FN) are poisonous mushrooms (actual class 1)
# that were misclassified as edible (predicted class 0).
# This corresponds to the element at cm[1, 0] in the confusion matrix.
false_negatives = cm[1, 0]

# Total actual poisonous mushrooms in the test set
# This is the sum of False Negatives (FN) and True Positives (TP)
# which corresponds to the entire 'Actual Poisonous' row (row index 1).
total_actual_poisonous = cm[1, 0] + cm[1, 1]

# Calculate FNR
if total_actual_poisonous > 0:
    false_negative_rate = false_negatives / total_actual_poisonous
else:
    # If there are no actual poisonous mushrooms in the test set,
    # FNR is technically 0 as no misclassifications are possible.
    false_negative_rate = 0.0

print(f"\nNumber of False Negatives (Poisonous misclassified as Edible): {false_negatives}")
print(f"Total actual Poisonous mushrooms in test set: {total_actual_poisonous}")
print(f"False Negative Rate (FNR): {false_negative_rate:.4f}")

# 12. Add print statements to issue a severe warning if FNR is greater than 0
if false_negative_rate > 0:
    print("\n" + "="*80)
    print("!!! SEVERE MYCOGUARD SAFETY WARNING !!!".center(80))
    print("--- PROTOCOL VIOLATION: POISONOUS MUSHROOMS MISCLASSIFIED ---".center(80))
    print("\nThis model has incorrectly identified poisonous mushrooms as edible.".center(80))
    print(f"False Negative Rate (Poisonous -> Edible): {false_negative_rate:.4f}".center(80))
    print(f"Number of such critical misclassifications: {false_negatives}".center(80))
    print("\nThis poses an UNACCEPTABLE AND CRITICAL RISK to human safety.".center(80))
    print("Deployment of this model in its current state is STRICTLY PROHIBITED.".center(80))
    print("IMMEDIATE retraining, re-evaluation, and improvement are REQUIRED.".center(80))
    print("="*80)
else:
    print("\nMycoGuard Safety Check Passed: No poisonous mushrooms were misclassified as edible (FNR = 0).")
    print("This indicates a robust safety performance regarding edible classification.")

print("\n" + "="*80)
print("--- MycoGuard Safety Audit Complete ---".center(80))
print("="*80)
```

--------------------------------------------------


--- 🍄 MycoGuard Generative Workflow Complete ---


In [ ]:
#use it if needed to kill port
!fuser -k 8000/tcp

In [10]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import nest_asyncio
from pyngrok import ngrok
import threading

nest_asyncio.apply()

app = FastAPI()

class PromptRequest(BaseModel):
    query: str

# Create an API endpoint
@app.post("/generate")
async def generate_code(request: PromptRequest):
    print(f"Received request from frontend: {request.query}")

    # Call the Router Agent to determine task routing
    # Ensure router_agent and session_service are already initialized in memory
    router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
    chosen_route = await run_agent_query(router_agent, request.query, router_session, my_user_id, is_router=True)
    chosen_route = chosen_route.strip().replace("'", "")

    final_output = ""

    if chosen_route == "mycoguard_sequential_combo":
        # Run the three agents sequentially and combine their generated code

        auditor_session = await session_service.create_session(app_name=data_auditor_agent.name, user_id=my_user_id)
        auditor_code = await run_agent_query(
            data_auditor_agent,
            "Please write the Python code to load 'mushroom(1).csv', handle missing values, and encode categorical variables.",
            auditor_session,
            my_user_id
        )

        forge_session = await session_service.create_session(app_name=classifier_forge_agent.name, user_id=my_user_id)
        forge_code = await run_agent_query(
            classifier_forge_agent,
            f"Based on this preprocessing code:\n{auditor_code}\nWrite the next part of the script to train a Random Forest classifier.",
            forge_session,
            my_user_id
        )

        validator_session = await session_service.create_session(app_name=safety_validator_agent.name, user_id=my_user_id)
        validator_code = await run_agent_query(
            safety_validator_agent,
            f"Based on the training code:\n{forge_code}\nWrite the final part of the script to output a Confusion Matrix and strictly check for False Negatives.",
            validator_session,
            my_user_id
        )

        # Combine the generated code sections
        final_output = (
            f"# --- Data Auditor ---\n{auditor_code}\n\n"
            f"# --- Classifier Forge ---\n{forge_code}\n\n"
            f"# --- Safety Validator ---\n{validator_code}"
        )

    else:
        final_output = "Sorry, the MycoGuard safety pipeline was not triggered."

    print("Code generation completed. Returning results to frontend...")
    return {"status": "success", "generated_code": final_output}

# Insert your NGROK authentication token here
NGROK_AUTH_TOKEN = "paste your NGROK token here "
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start the Ngrok tunnel
public_url = ngrok.connect(8000).public_url
print(f"Public API endpoint: {public_url}/generate")
print("Server started and waiting for frontend requests...")

# Run FastAPI server in a background thread
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)
).start()


Public API endpoint: https://julieann-unfumigated-denotatively.ngrok-free.dev/generate
Server started and waiting for frontend requests...
